# ReviewPulse serving benchmark (vLLM on a Colab GPU)

Runs an open 7-8B model with vLLM and scores it on the ReviewPulse workload: theme-tagging F1 on the SemEval-2014 test set, and output tokens/s. Use **Runtime > Change runtime type > GPU** (T4 on free Colab, L4 on Colab Pro). Takes about 30-45 minutes.

Copy the printed row into `results/benchmarks.csv` in the repo (or download the file at the end).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install vllm httpx

In [ ]:
# If the repo is still private, skip this cell: upload bench/bench_reviewpulse.py with the Files panel,
# then run: !mkdir -p bench results && mv bench_reviewpulse.py bench/
!git clone -q https://github.com/iashutoshparida/llm-serving-cost-pricing.git
%cd llm-serving-cost-pricing

## Models
AWQ 4-bit models fit a 16 GB T4. On an L4 (24 GB) also run the full-precision model to see what quantisation costs in quality.

In [ ]:
MODELS = [
    ("Qwen/Qwen2.5-7B-Instruct-AWQ", "awq"),
    ("hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4", "awq"),
    # ("Qwen/Qwen2.5-7B-Instruct", "none"),   # L4 only
]
GPU = "T4 (Colab)"   # change to "L4 (Colab Pro)" if that is what nvidia-smi shows

In [ ]:
import subprocess, time, httpx

def serve(model, quant):
    cmd = ["vllm", "serve", model, "--max-model-len", "4096", "--gpu-memory-utilization", "0.90", "--port", "8000"]
    if quant != "none":
        cmd += ["--quantization", quant]
    if "T4" in GPU:
        cmd += ["--dtype", "half"]
    proc = subprocess.Popen(cmd, stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(180):
        try:
            if httpx.get("http://localhost:8000/health", timeout=2).status_code == 200:
                return proc
        except Exception:
            pass
        time.sleep(5)
    raise RuntimeError(open("vllm.log").read()[-3000:])

for model, quant in MODELS:
    proc = serve(model, quant)
    !python bench/bench_reviewpulse.py --model {model} --quant {quant} --gpu "{GPU}" --concurrency 16
    proc.terminate(); proc.wait(); time.sleep(10)

In [ ]:
import pandas as pd
pd.read_csv("results/benchmarks.csv")

In [ ]:
from google.colab import files
files.download("results/benchmarks.csv")